In [2]:
!pip install onnx

In [3]:
!pip install onnxscript

In [4]:
!pip install onnxruntime

In [5]:
!pip install torch torchvision numpy pillow

### Download a small Model(Baseline)

In [2]:
import torch
from torchvision import models
from PIL import Image
import onnxruntime as ort

weights = models.MobileNet_V2_Weights.DEFAULT   
#selects the recommended pretrained weights for MobileNet V2
model = models.mobilenet_v2(weights=weights)    
model.eval()                                   
#tells pytroch to su eteh mdoel in evauation mode 

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

In [3]:
# Searching for dataset
labels = weights.meta["categories"]
print(len(labels))        # 1000
print(labels[:20])        # first 20 classes

print([l for l in labels if "dog" in l.lower() or "retriever" in l.lower()])

1000
['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead', 'electric ray', 'stingray', 'cock', 'hen', 'ostrich', 'brambling', 'goldfinch', 'house finch', 'junco', 'indigo bunting', 'robin', 'bulbul', 'jay', 'magpie', 'chickadee']
['Maltese dog', 'flat-coated retriever', 'curly-coated retriever', 'golden retriever', 'Labrador retriever', 'Chesapeake Bay retriever', 'Old English sheepdog', 'Shetland sheepdog', 'Greater Swiss Mountain dog', 'Bernese mountain dog', 'French bulldog', 'Eskimo dog', 'African hunting dog', 'dogsled', 'hotdog']


In [45]:
# Making inference
preprocess = weights.transforms()    # correct resize + normalize so image can go inside the model
img = Image.open("test.jpeg").convert("RGB")
#prepare image for the model
x = preprocess(img).unsqueeze(0)                # shape [1, 3, 224, 224]
#no unncessary gradient cal during pred
with torch.no_grad():
    out = model(x)

top = out.argmax(1).item()
print(weights.meta["categories"][top])

espresso


### Record size, speed, and accuracy(Baseline)

In [5]:
#(Downloading Imagenette) for accuracy calculation

import urllib.request, tarfile

url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
urllib.request.urlretrieve(url, "imagenette.tgz")
tarfile.open("imagenette.tgz").extractall(".")
print("Done")

C:\Users\Muhammad Tahir\AppData\Local\Temp\ipykernel_3508\1226184569.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open("imagenette.tgz").extractall(".")


Done


In [46]:

folder_to_idx = {
 "n01440764": 0,   "n02102040": 217, "n02979186": 482, "n03000684": 491,
 "n03028079": 497, "n03394916": 566, "n03417042": 569, "n03425413": 571,
 "n03445777": 574, "n03888257": 701}

labels = weights.meta["categories"]
for f, i in folder_to_idx.items():
    print(f, "->", labels[i])

n01440764 -> tench
n02102040 -> English springer
n02979186 -> cassette player
n03000684 -> chain saw
n03028079 -> church
n03394916 -> French horn
n03417042 -> garbage truck
n03425413 -> gas pump
n03445777 -> golf ball
n03888257 -> parachute


In [47]:
# SIZE
import os, torch
torch.save(model.state_dict(), "mobilenetv2_fp32.pth")
print(f"File size: {os.path.getsize('mobilenetv2_fp32.pth') / 1e6:.1f} MB")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

File size: 14.3 MB
Parameters: 3.50 M


In [48]:
# SPEED

import time, numpy as np

with torch.no_grad():
    for _ in range(10):                  
        model(x)

    times = []
    for _ in range(100):
        t = time.perf_counter()
        model(x)
        times.append((time.perf_counter() - t) * 1000)

print(f"Mean: {np.mean(times):.1f} ms")
print(f"FPS: {1000 / np.mean(times):.1f}")

Mean: 71.4 ms
FPS: 14.0


In [58]:
# Accuracy

import glob
from PIL import Image

correct = total = 0
with torch.no_grad():
    for _ in range(10):
        model(x_torch)

    times = []
    for _ in range(100):
        t = time.perf_counter()
        model(x_torch)
        times.append((time.perf_counter() - t) * 1000)

print(f"Mean: {np.mean(times):.1f} ms")
print(f"FPS: {1000 / np.mean(times):.1f}")

Mean: 72.3 ms
FPS: 13.8


In [57]:
import time, numpy as np

img = Image.open("test.jpeg").convert("RGB")
x_torch = preprocess(img).unsqueeze(0)               # for PyTorch
x_onnx = x_torch.numpy().astype(np.float32)          # same data for ONNX

In [12]:
# which classes lose points

from collections import Counter
wrong = Counter()
with torch.no_grad():
    for folder, idx in folder_to_idx.items():
        for f in glob.glob(f"imagenette2-160/val/{folder}/*.JPEG")[:20]:
            img = Image.open(f).convert("RGB")
            pred = model(preprocess(img).unsqueeze(0)).argmax(1).item()
            if pred != idx:
                wrong[labels[idx]] += 1
print(wrong)

Counter({'church': 16, 'gas pump': 10, 'English springer': 8, 'cassette player': 8, 'chain saw': 7, 'French horn': 7, 'garbage truck': 6, 'golf ball': 3, 'parachute': 3, 'tench': 2})


### Export Model on ONNX

In [17]:
torch.onnx.export(
    model, x, "mobilenetv2.onnx",
    input_names=["input"], output_names=["output"],
    opset_version=18
)
print("Exported to ONNX")

[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported to ONNX


In [21]:
import onnx
onnx.checker.check_model(onnx.load("mobilenetv2.onnx"))

### Load ONNX Model using ONNX Runtime

In [25]:
# Check available exceution providers
print(ort.get_available_providers())

['AzureExecutionProvider', 'CPUExecutionProvider']


In [23]:
sess = ort.InferenceSession("mobilenetv2.onnx", providers=["CPUExecutionProvider"])
inp_name = sess.get_inputs()[0].name
print("Input:", inp_name, sess.get_inputs()[0].shape)

Input: input [1, 3, 224, 224]


### Comparing Baseline Pytorch Model and ONNX Runtime Outputs

In [60]:
x = torch.randn(1, 3, 224, 224)
model.eval()                                  
with torch.no_grad():
    torch_out = model(x_torch).numpy()
onnx_out = sess.run(None, {inp_name: x_onnx})[0]
np.testing.assert_allclose(torch_out, onnx_out, rtol=1e-3, atol=1e-4)
print("Outputs match: export is correct")

Outputs match: export is correct


### Re-measure size, speed, and accuracy for ONNX FP32

In [55]:
# Size
import os
import onnx
from onnx import numpy_helper

path = "mobilenetv2.onnx"

size = os.path.getsize(path)
if os.path.exists(path + ".data"):
    size += os.path.getsize(path + ".data")
print(f"File size (MB): {size / 1e6:.2f}")

m = onnx.load(path, load_external_data=True)
params = sum(numpy_helper.to_array(t).size for t in m.graph.initializer)
print(f"Parameters: {params / 1e6:.2f} M")

File size (MB): 14.21
Parameters: 3.49 M


In [59]:
# Speed
inp = sess.get_inputs()[0].name
for _ in range(10):                      # warm-up
    sess.run(None, {inp: x_onnx})
times = []
for _ in range(100):
    t = time.perf_counter()
    sess.run(None, {inp: x_onnx})
    times.append((time.perf_counter() - t) * 1000)
mean = np.mean(times)
print(f"Mean: {mean:.1f} ms | FPS: {1000/mean:.1f}")

Mean: 15.3 ms | FPS: 65.3


In [61]:
# Accuracy
import glob
correct = total = 0
for folder, idx in folder_to_idx.items():
    files = glob.glob(f"imagenette2-160/val/{folder}/*.JPEG")[:20]
    for f in files:
        img = Image.open(f).convert("RGB")
        xi = preprocess(img).unsqueeze(0).numpy().astype(np.float32)  
        out = sess.run(None, {inp: xi})[0]
        pred = int(out.argmax(axis=1)[0])
        correct += (pred == idx)
        total += 1

print(f"Accuracy: {correct/total*100:.1f}% ({correct}/{total})")

Accuracy: 65.0% (130/200)


### Optimize(Post-Training Static Quantization)

In [62]:
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantFormat, QuantType
)

# Calibration: ~50 real images show ONNX Runtime the typical value ranges
class ImgReader(CalibrationDataReader):
    def __init__(self):
        files = []
        for folder in folder_to_idx:
            files += glob.glob(f"imagenette2-160/train/{folder}/*.JPEG")[:5]
            #calibration
        self.data = iter([
            {inp_name: preprocess(Image.open(f).convert("RGB")).unsqueeze(0).numpy().astype(np.float32)}
            for f in files
        ])
    def get_next(self):
        return next(self.data, None)

quantize_static(
    "mobilenetv2.onnx", "mobilenetv2_int8.onnx",
    calibration_data_reader=ImgReader(),
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QUInt8,
    per_channel=True,
)
print("Quantized")

Quantized


### Re-measure size, speed, and accuracy for ONNX INT8

In [63]:
import os
q_path = "mobilenetv2_int8.onnx"
qsess = ort.InferenceSession(q_path, providers=["CPUExecutionProvider"])
qinp = qsess.get_inputs()[0].name

# Size
size = os.path.getsize(q_path)
if os.path.exists(q_path + ".data"):
    size += os.path.getsize(q_path + ".data")
print(f"File size (MB): {size/1e6:.2f}")

# Speed 
for _ in range(10):
    qsess.run(None, {qinp: x_onnx})
times = []
for _ in range(100):
    t = time.perf_counter()
    qsess.run(None, {qinp: x_onnx})
    times.append((time.perf_counter() - t) * 1000)
print(f"Mean: {np.mean(times):.1f} ms | FPS: {1000/np.mean(times):.1f}")

# Accuracy
correct = total = 0
for folder, idx in folder_to_idx.items():
    for f in glob.glob(f"imagenette2-160/val/{folder}/*.JPEG")[:20]:
        xi = preprocess(Image.open(f).convert("RGB")).unsqueeze(0).numpy().astype(np.float32)
        pred = int(qsess.run(None, {qinp: xi})[0].argmax(axis=1)[0])
        correct += (pred == idx)
        total += 1
print(f"Accuracy: {correct/total*100:.1f}% ({correct}/{total})")

File size (MB): 4.02
Mean: 7.9 ms | FPS: 125.9
Accuracy: 65.0% (130/200)


In [ ]:
### Deploy to the edge device

In [65]:
so = ort.SessionOptions()
so.intra_op_num_threads = 1
so.inter_op_num_threads = 1
edge_sess = ort.InferenceSession("mobilenetv2_int8.onnx", so, providers=["CPUExecutionProvider"])
edge_inp = edge_sess.get_inputs()[0].name

for _ in range(10):                      # warm-up
    edge_sess.run(None, {edge_inp: x_onnx})
times = []
for _ in range(100):
    t = time.perf_counter()
    edge_sess.run(None, {edge_inp: x_onnx})
    times.append((time.perf_counter() - t) * 1000)
print(f"Edge (1 thread) Mean: {np.mean(times):.1f} ms | FPS: {1000/np.mean(times):.1f}")

Edge (1 thread) Mean: 16.0 ms | FPS: 62.5
